# Implementación del Modelo de Regresión Darvas & Schepp (2024)
Este notebook implementa el modelo de corrección de errores (ECM) descrito en el artículo:
*Exchange rates and fundamentals: Forecasting with long maturity forward rates* (Journal of International Money and Finance, 2024).

**Nota importante:** Esta implementación corrige el sesgo de anticipación (look-ahead bias) asegurando que el modelo se entrene solo con datos disponibles en el momento del pronóstico.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Obtención de datos históricos
ticker = "EURUSD=X"
df = yf.download(ticker, start="2015-01-01", interval="1d")
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
df.columns = [col.lower() for col in df.columns]
df.index.name = "time"
df = df[~df.index.duplicated(keep="first")]
df["s"] = np.log(df["close"])
df.tail()

In [ ]:
# Definición del Proxy para el Fundamental Equilibrium
# Usamos una media móvil de 750 días como proxy del forward rate a largo plazo
W = 750
df["f_proxy"] = df["s"].rolling(window=W).mean()
df["x"] = df["f_proxy"] - df["s"] # El gap o término de corrección de errores

# Horizonte de pronóstico (1 mes ~ 22 días hábiles)
H = 22
df["target"] = df["s"].shift(-H) - df["s"]
df = df.dropna(subset=["x"])

In [ ]:
# Estimación Recursiva Out-of-Sample (OOS) - Corregido
split_idx = int(len(df) * 0.8)
predictions, actuals, dates = [], [], []

print("Ejecutando pronósticos recursivos (sin look-ahead bias)...")
for i in range(split_idx, len(df) - H):
    # El último índice de entrenamiento usable j debe cumplir j + H <= i
    last_train_idx = i - H
    if last_train_idx < 100: continue
    
    current_train = df.iloc[:last_train_idx+1].dropna(subset=["target"])
    model = smf.ols(formula="target ~ x", data=current_train).fit()
    
    x_t = df.iloc[i]["x"]
    pred_change = model.params["Intercept"] + model.params["x"] * x_t
    
    predictions.append(pred_change)
    actuals.append(df.iloc[i]["target"])
    dates.append(df.index[i])

results = pd.DataFrame({"Actual": actuals, "Model_Pred": predictions, "RW_Pred": 0}, index=dates)

In [ ]:
rmse_model = np.sqrt(np.mean((results["Actual"] - results["Model_Pred"])**2))
rmse_rw = np.sqrt(np.mean((results["Actual"])**2))

print(f"RMSE Modelo: {rmse_model:.6f}")
print(f"RMSE Random Walk: {rmse_rw:.6f}")
print(f"Ratio RMSE (Modelo/RW): {rmse_model/rmse_rw:.4f}")

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(results["Actual"].cumsum(), label="Actual (Cumulativo)")
plt.plot(results["Model_Pred"].cumsum(), label="Modelo (Cumulativo)", linestyle="--")
plt.title("Rendimiento del Modelo ECM vs Actual (Corregido)")
plt.legend()
plt.show()